# Multi-Head Attention

## 왜 필요한가?

- 하나의 Head는 한 가지 관계만 학습하기 쉽다.
- 여러 Head가 서로 다른 관계를 병렬로 학습한다.
- Concat 후 Linear Layer로 정보를 통합한다.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

In [2]:
import numpy as np
import torch
from torch import nn
from src.attention import scaled_dot_product_attention

attention.py loaded


In [19]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=512, num_heads=8):
        super().__init__()

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        assert d_model % num_heads == 0

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, Q, K, V):
        Q = self.W_q(Q)
        K = self.W_k(K)
        V = self.W_v(V)

        batch_size = Q.size(0)
        seq_len = Q.size(1)

        Q = Q.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        K = K.reshape(batch_size, seq_len, self.num_heads, self.head_dim)
        V = V.reshape(batch_size, seq_len, self.num_heads, self.head_dim)

        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        
        scores, attn_weights, output = scaled_dot_product_attention(Q, K, V)

        output = output.transpose(1, 2)
        output = output.reshape(batch_size, seq_len, self.num_heads * self.head_dim)
        output = self.W_o(output)

        return output

Q = torch.randn(32, 10, 512)
K = torch.randn(32, 10, 512)
V = torch.randn(32, 10, 512)

model = MultiHeadAttention()
output = model(Q, K, V)
print(output.shape)


torch.Size([32, 10, 512])
